# LeetCode 711 — Number of Distinct Islands II bằng Python

- Link: https://leetcode.com/problems/number-of-distinct-islands-ii/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: DFS/BFS trên lưới, Hình học trên toạ độ (xoay, lật, tịnh tiến), Dạng chuẩn (canonical form), Hashing bằng `set`

> **Ghi chú Premium:** Đây là bài dành cho tài khoản Premium. Phần đề bài, ví dụ và ràng buộc dưới đây được viết lại theo hiểu biết về bài. Nếu bạn có tài khoản Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc trên trang LeetCode.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numDistinctIslands2(self, grid: List[List[int]]) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bài 694 (*Number of Distinct Islands*) coi hai đảo giống nhau nếu **tịnh tiến** được thành nhau. Bài này mở rộng: hai đảo giống nhau nếu biến được thành nhau bằng **xoay** (90°, 180°, 270°) **và/hoặc lật** (gương ngang, gương dọc) — tổng cộng có **8 phép biến đổi** của hình vuông.

Ý tưởng then chốt rất đáng học: thay vì so từng cặp đảo, ta tính cho mỗi đảo một **dạng chuẩn (canonical form)** — một "chữ ký" giống hệt nhau cho mọi đảo cùng hình dạng. Đếm số chữ ký khác nhau là xong.

Sau notebook này bạn sẽ:

1. Tìm các đảo (thành phần liên thông 4 hướng) bằng DFS/BFS không đệ quy.
2. Biểu diễn 8 phép xoay/lật bằng công thức toạ độ `(r, c) → (±r, ±c)` hoặc `(±c, ±r)`.
3. Chuẩn hoá tịnh tiến (đưa về góc trên-trái) và chuẩn hoá thứ tự (sắp xếp).
4. Xây dạng chuẩn = **nhỏ nhất** trong 8 phiên bản đã chuẩn hoá, và đếm bằng `set`.
5. Kiểm chứng bằng brute force so từng cặp và bằng **tính chất bất biến**: xoay/lật cả lưới thì đáp án không đổi.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho lưới nhị phân `grid` kích thước `m × n`. Một **đảo** là nhóm các ô `1` liên thông theo 4 hướng (trên, dưới, trái, phải). Xung quanh lưới coi như là nước.

Hai đảo được coi là **giống nhau** nếu có thể biến đảo này thành đảo kia bằng cách **xoay** (90, 180 hoặc 270 độ) và/hoặc **lật** (đối xứng gương trái-phải hoặc trên-dưới), kết hợp với **tịnh tiến**. Trả về **số đảo khác nhau**.

**Ví dụ 1**

```text
Input:
11000
10000
00001
00011
Output: 1
Giải thích: đảo trên-trái {(0,0),(0,1),(1,0)} xoay 180° thành đảo dưới-phải {(2,4),(3,3),(3,4)}.
```

**Ví dụ 2**

```text
Input:
11100
10001
01001
01110
Output: 2
Giải thích: hai đảo hình chữ L (4 ô) là ảnh gương của nhau → giống nhau;
            đảo dọc 2 ô {(1,4),(2,4)} là một hình khác → tổng 2 loại.
```

**Ràng buộc (theo hiểu biết, nên đối chiếu):**

- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 50`
- `grid[i][j]` là `0` hoặc `1`.

In [1]:
def parse(rows):
    return [[int(ch) for ch in row] for row in rows]

ex1 = parse(["11000", "10000", "00001", "00011"])
ex2 = parse(["11100", "10001", "01001", "01110"])

def show(grid):
    for row in grid:
        print(''.join('#' if v else '.' for v in row))

show(ex1); print()
show(ex2)

##...
#....
....#
...##

###..
#...#
.#..#
.###.


## 3. Kiến thức nền

### 3.1. Tìm đảo bằng DFS dùng ngăn xếp (không đệ quy)

Đảo có thể tới 2500 ô; đệ quy sâu 2500 tầng dễ vượt giới hạn đệ quy của Python (mặc định 1000). Dùng **ngăn xếp tường minh** (list + `pop`) cho an toàn.

In [2]:
def find_islands(grid):
    """Trả về danh sách đảo, mỗi đảo là list các ô (r, c)."""
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    islands = []
    for sr in range(m):
        for sc in range(n):
            if grid[sr][sc] and not seen[sr][sc]:
                seen[sr][sc] = True
                stack, cells = [(sr, sc)], []
                while stack:
                    r, c = stack.pop()
                    cells.append((r, c))
                    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and not seen[nr][nc]:
                            seen[nr][nc] = True
                            stack.append((nr, nc))
                islands.append(cells)
    return islands

for isl in find_islands(ex2):
    print(sorted(isl))

[(0, 0), (0, 1), (0, 2), (1, 0)]
[(1, 4), (2, 4)]
[(2, 1), (3, 1), (3, 2), (3, 3)]


### 3.2. Chuẩn hoá tịnh tiến

Hai đảo chỉ khác vị trí thì sau khi **trừ đi (min hàng, min cột)** sẽ có cùng tập toạ độ. Sau đó **sắp xếp** để thứ tự liệt kê ô không ảnh hưởng.

In [3]:
def normalize(cells):
    r0 = min(r for r, _ in cells)
    c0 = min(c for _, c in cells)
    return tuple(sorted((r - r0, c - c0) for r, c in cells))

a = [(0, 0), (0, 1), (1, 0)]
b = [(5, 7), (4, 7), (4, 8)]          # cùng hình, khác vị trí và thứ tự
print(normalize(a), normalize(b), normalize(a) == normalize(b))

((0, 0), (0, 1), (1, 0)) ((0, 0), (0, 1), (1, 0)) True


### 3.3. Tám phép xoay/lật bằng toạ độ

Đặt mỗi ô là điểm `(x, y)`. Nhóm đối xứng của hình vuông có 8 phần tử:

| Phép | Công thức |
|---|---|
| giữ nguyên | `(x, y)` |
| lật dọc (đảo cột) | `(x, -y)` |
| lật ngang (đảo hàng) | `(-x, y)` |
| xoay 180° | `(-x, -y)` |
| lật qua đường chéo chính | `(y, x)` |
| xoay 90° (một chiều) | `(y, -x)` |
| xoay 90° (chiều kia) | `(-y, x)` |
| lật qua đường chéo phụ | `(-y, -x)` |

Toạ độ âm không sao — bước chuẩn hoá tịnh tiến sẽ đưa về không âm.

In [4]:
TRANSFORMS = [
    lambda x, y: (x, y),   lambda x, y: (x, -y),
    lambda x, y: (-x, y),  lambda x, y: (-x, -y),
    lambda x, y: (y, x),   lambda x, y: (y, -x),
    lambda x, y: (-y, x),  lambda x, y: (-y, -x),
]

def draw_cells(cells):
    cells = set(normalize(cells))
    H = max(r for r, _ in cells) + 1
    W = max(c for _, c in cells) + 1
    return [''.join('#' if (r, c) in cells else '.' for c in range(W)) for r in range(H)]

L = [(0, 0), (1, 0), (2, 0), (2, 1)]     # chữ L
pics = [draw_cells([t(x, y) for x, y in L]) for t in TRANSFORMS]
for row in range(3):
    print('   '.join(p[row].ljust(3) if row < len(p) else '   ' for p in pics))

#.    .#    ##    ##    ###   ###   ..#   #..
#.    .#    #.    .#    ..#   #..   ###   ###
##    ##    #.    .#                         


Tám hình trên là 8 phiên bản của chữ L (một số hình có thể trùng nhau với những đảo đối xứng — ví dụ hình vuông 2×2 có cả 8 phiên bản giống hệt nhau).

### 3.4. Xoay ma trận bằng `zip`

Một cách khác để xoay: biểu diễn đảo bằng ma trận nhỏ (khung bao), xoay 90° bằng `list(zip(*M[::-1]))`, lật bằng đảo từng hàng `row[::-1]`.

In [5]:
M = ["#.", "#.", "##"]
rot = [''.join(t) for t in zip(*M[::-1])]
flip = [row[::-1] for row in M]
print("gốc :", M)
print("xoay:", rot)
print("lật :", flip)

gốc : ['#.', '#.', '##']
xoay: ['###', '#..']
lật : ['.#', '.#', '##']


### 3.5. So sánh tuple và `min`

Tuple so sánh theo thứ tự từ điển, nên `min(list_of_tuples)` luôn cho **một** phần tử xác định. Đó là cách chọn "đại diện chuẩn" trong 8 phiên bản.

In [6]:
versions = [((0, 1), (1, 0)), ((0, 0), (1, 1)), ((0, 0), (0, 1))]
print(min(versions))

((0, 0), (0, 1))


## 4. Ý tưởng nền tảng / trực giác

### 4.1. "Giống nhau" là một quan hệ tương đương

Hai đảo giống nhau nếu một trong 8 phép biến đổi (kèm tịnh tiến) đưa đảo này trùng đảo kia. Quan hệ này phản xạ, đối xứng, bắc cầu (vì 8 phép tạo thành một **nhóm**: hợp hai phép vẫn là một phép trong 8, mỗi phép có phép ngược). Vậy các đảo chia thành các **lớp tương đương**; đáp án = số lớp.

### 4.2. Tính tay ví dụ 1

Đảo A = `{(0,0),(0,1),(1,0)}`; đảo B = `{(2,4),(3,3),(3,4)}`.

- Chuẩn hoá tịnh tiến B: trừ `(2, 3)` → `{(0,1),(1,0),(1,1)}`.
- Xoay 180° A: `(x, y) → (-x, -y)` → `{(0,0),(0,-1),(-1,0)}` → trừ `(-1, -1)` → `{(1,1),(1,0),(0,1)}` = B đã chuẩn hoá ✓.

Vậy A và B cùng lớp → đáp án 1.

### 4.3. Dạng chuẩn

Với mỗi đảo, tính 8 phiên bản (biến đổi → chuẩn hoá tịnh tiến → sắp xếp) và lấy **phiên bản nhỏ nhất**. Hai đảo cùng lớp có **cùng tập 8 phiên bản** (vì 8 phép là một nhóm) → cùng phiên bản nhỏ nhất. Hai đảo khác lớp thì tập phiên bản rời nhau → dạng chuẩn khác nhau.

In [7]:
A = [(0, 0), (0, 1), (1, 0)]
B = [(2, 4), (3, 3), (3, 4)]
vers_A = sorted({normalize([t(x, y) for x, y in A]) for t in TRANSFORMS})
vers_B = sorted({normalize([t(x, y) for x, y in B]) for t in TRANSFORMS})
print("các phiên bản của A:", vers_A)
print("tập phiên bản A == B?", vers_A == vers_B, "| dạng chuẩn:", vers_A[0])

các phiên bản của A: [((0, 0), (0, 1), (1, 0)), ((0, 0), (0, 1), (1, 1)), ((0, 0), (1, 0), (1, 1)), ((0, 1), (1, 0), (1, 1))]
tập phiên bản A == B? True | dạng chuẩn: ((0, 0), (0, 1), (1, 0))


## 5. Cách 1 — Brute Force: so từng cặp đảo bằng xoay/lật ma trận

**Ý tưởng.** Biểu diễn mỗi đảo bằng **ma trận khung bao** (list chuỗi `#`/`.`). Giữ danh sách "đại diện" các loại đảo đã gặp. Với mỗi đảo mới, sinh 8 ma trận (4 phép xoay của bản gốc và 4 phép xoay của bản lật); nếu có ma trận nào **bằng** ma trận của một đại diện → cùng loại; nếu không → thêm đại diện mới.

Cách này dùng phép xoay ma trận (khác hẳn công thức toạ độ ở các cách sau) nên rất hợp làm **trọng tài độc lập**.

In [8]:
def to_matrix(cells):
    r0 = min(r for r, _ in cells); c0 = min(c for _, c in cells)
    H = max(r for r, _ in cells) - r0 + 1
    W = max(c for _, c in cells) - c0 + 1
    s = {(r - r0, c - c0) for r, c in cells}
    return tuple(''.join('1' if (r, c) in s else '0' for c in range(W)) for r in range(H))

def rotate(M):
    return tuple(''.join(t) for t in zip(*M[::-1]))

def mirror(M):
    return tuple(row[::-1] for row in M)

def all_orientations(M):
    out = []
    for base in (M, mirror(M)):
        cur = base
        for _ in range(4):
            out.append(cur)
            cur = rotate(cur)
    return out

def distinct_islands_brute(grid):
    reps = []                                   # ma trận đại diện của mỗi loại
    for cells in find_islands(grid):
        M = to_matrix(cells)
        if not any(O == R for O in all_orientations(M) for R in reps):
            reps.append(M)
    return len(reps)

print(distinct_islands_brute(ex1), distinct_islands_brute(ex2))

1 2


**Độ phức tạp.** Gọi `I` là số đảo, `S` là kích thước đảo. Mỗi đảo so với tối đa `I` đại diện × 8 hướng, mỗi lần so `O(S)` (ma trận khung bao, có thể lớn hơn `S`). Tổng `O(I² · 8 · B)` với `B` là diện tích khung bao. Với lưới 50×50, `I` có thể tới ~1250 đảo một ô → `~10^7` phép so sánh nhỏ — chậm nhưng chạy được; trường hợp xấu hơn (nhiều đảo vừa) sẽ rất chậm.

**Nút thắt.** So **từng cặp** là `O(I²)`. Nếu mỗi đảo có một "chữ ký" duy nhất cho cả lớp, ta chỉ cần cho chữ ký vào `set` → `O(I)` lần băm.

## 6. Bước đột phá

**Insight:** dùng **dạng chuẩn** thay cho so sánh cặp.

> `canon(đảo) = min{ normalize(T(đảo)) : T ∈ 8 phép }`

**Quy tắc "nếu… thì…":**

- **Nếu** hai đảo giống nhau (tồn tại `T` với `T(A) ≅ B` sau tịnh tiến) **thì** `{normalize(T'(A))} = {normalize(T'(B))}` (vì tập 8 phép đóng với phép hợp) ⇒ `canon(A) = canon(B)`.
- **Nếu** `canon(A) = canon(B)` **thì** có `T1, T2` với `normalize(T1(A)) = normalize(T2(B))` ⇒ `T2⁻¹ ∘ T1` biến `A` thành `B` (sai khác tịnh tiến) ⇒ giống nhau.

Vậy **đáp án = số dạng chuẩn khác nhau** = `len(set(canon(đảo) for đảo in islands))`.

**Bất biến cần giữ khi chuẩn hoá:** mọi phiên bản phải được (1) tịnh tiến về cùng một mốc (min hàng = 0, min cột = 0) và (2) sắp xếp — thiếu một trong hai, cùng một hình dạng sẽ cho ra hai tuple khác nhau.

**Chuẩn hoá tịnh tiến có hai lựa chọn phổ biến** (đều đúng): trừ `(min r, min c)`, hoặc sắp xếp rồi trừ **điểm đầu tiên**. Ta dùng lựa chọn thứ nhất cho dễ hình dung.

## 7. Các cách tiếp theo

Tổng cộng 3 cách:

| Cách | Ý tưởng | Time |
|---|---|---|
| 1 | So từng cặp đảo bằng xoay/lật ma trận | `O(I² · 8 · B)` |
| 2 | Dạng chuẩn = min 8 tuple toạ độ đã chuẩn hoá; đếm bằng `set` | `O(N log N)` với `N = m·n` |
| 3 | Dạng chuẩn = min 8 **chuỗi ma trận** khung bao; BFS tìm đảo | `O(Σ 8 · B)` |

### 7.1. Cách 2 — Dạng chuẩn bằng tuple toạ độ

In [9]:
def canonical(cells):
    best = None
    for t in TRANSFORMS:
        pts = [t(r, c) for r, c in cells]
        r0 = min(p[0] for p in pts)
        c0 = min(p[1] for p in pts)
        version = tuple(sorted((p[0] - r0, p[1] - c0) for p in pts))
        if best is None or version < best:
            best = version
    return best

def distinct_islands_canon(grid):
    return len({canonical(cells) for cells in find_islands(grid)})

print(distinct_islands_canon(ex1), distinct_islands_canon(ex2))
for cells in find_islands(ex2):
    print(sorted(cells), "-> dạng chuẩn", canonical(cells))

1 2
[(0, 0), (0, 1), (0, 2), (1, 0)] -> dạng chuẩn ((0, 0), (0, 1), (0, 2), (1, 0))
[(1, 4), (2, 4)] -> dạng chuẩn ((0, 0), (0, 1))
[(2, 1), (3, 1), (3, 2), (3, 3)] -> dạng chuẩn ((0, 0), (0, 1), (0, 2), (1, 0))


Giải thích từng dòng của `canonical`:

- `pts = [t(r, c) ...]`: áp phép biến đổi lên mọi ô (có thể ra toạ độ âm).
- `r0, c0`: góc trên-trái của khung bao sau biến đổi.
- `version`: trừ góc rồi sắp xếp → phiên bản chuẩn hoá của hình sau phép `t`.
- `best`: giữ phiên bản nhỏ nhất (so sánh tuple theo từ điển).

**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def numDistinctIslands2(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        T = [(1, 1, False), (1, -1, False), (-1, 1, False), (-1, -1, False),
             (1, 1, True), (1, -1, True), (-1, 1, True), (-1, -1, True)]
        shapes = set()
        for sr in range(m):
            for sc in range(n):
                if grid[sr][sc] and not seen[sr][sc]:
                    seen[sr][sc] = True
                    stack, cells = [(sr, sc)], []
                    while stack:
                        r, c = stack.pop()
                        cells.append((r, c))
                        for nr, nc in ((r+1, c), (r-1, c), (r, c+1), (r, c-1)):
                            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and not seen[nr][nc]:
                                seen[nr][nc] = True
                                stack.append((nr, nc))
                    best = None
                    for a, b, swap in T:
                        pts = [(b * c, a * r) if swap else (a * r, b * c) for r, c in cells]
                        r0 = min(p[0] for p in pts); c0 = min(p[1] for p in pts)
                        v = tuple(sorted((x - r0, y - c0) for x, y in pts))
                        if best is None or v < best:
                            best = v
                    shapes.add(best)
        return len(shapes)
```

(Ở đây 8 phép được mã hoá bằng bộ `(dấu hàng, dấu cột, có đổi chỗ hay không)` — tương đương bảng ở mục 3.3.)

**Độ phức tạp.** Mỗi ô thuộc đúng một đảo; mỗi đảo kích thước `S` tốn `8 · S log S` (sắp xếp). Tổng `O(N log N)` với `N = m · n ≤ 2500`. Bộ nhớ `O(N)`.

### 7.2. Cách 3 — Dạng chuẩn bằng chuỗi ma trận khung bao + BFS

**Giải thích.** Thay vì tuple toạ độ, biểu diễn mỗi phiên bản bằng **chuỗi** "hàng1/hàng2/…" của ma trận khung bao. Sinh 8 phiên bản bằng `rotate` và `mirror` (mục 3.4). Dạng chuẩn = chuỗi nhỏ nhất. Tìm đảo bằng BFS (`deque`) cho đa dạng kỹ thuật.

Lưu ý: chuỗi phải chứa **dấu ngăn hàng** — nếu không, `"10" + "01"` và `"1001"` (một hàng) sẽ trùng nhau.

In [10]:
from collections import deque

def islands_bfs(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    for sr in range(m):
        for sc in range(n):
            if grid[sr][sc] and not seen[sr][sc]:
                seen[sr][sc] = True
                q, cells = deque([(sr, sc)]), []
                while q:
                    r, c = q.popleft()
                    cells.append((r, c))
                    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                        if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and not seen[nr][nc]:
                            seen[nr][nc] = True
                            q.append((nr, nc))
                yield cells

def canonical_string(cells):
    M = to_matrix(cells)
    return min('/'.join(O) for O in all_orientations(M))

def distinct_islands_string(grid):
    return len({canonical_string(cells) for cells in islands_bfs(grid)})

print(distinct_islands_string(ex1), distinct_islands_string(ex2))
for cells in islands_bfs(ex2):
    print(canonical_string(cells))

1 2
001/111
1/1
001/111


**Code nộp LeetCode (Cách 3):**

```python
from collections import deque

class Solution:
    def numDistinctIslands2(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        def to_matrix(cells):
            r0 = min(r for r, _ in cells); c0 = min(c for _, c in cells)
            H = max(r for r, _ in cells) - r0 + 1; W = max(c for _, c in cells) - c0 + 1
            s = {(r - r0, c - c0) for r, c in cells}
            return tuple(''.join('1' if (r, c) in s else '0' for c in range(W)) for r in range(H))
        rotate = lambda M: tuple(''.join(t) for t in zip(*M[::-1]))
        mirror = lambda M: tuple(row[::-1] for row in M)
        shapes = set()
        for sr in range(m):
            for sc in range(n):
                if grid[sr][sc] and not seen[sr][sc]:
                    seen[sr][sc] = True
                    q, cells = deque([(sr, sc)]), []
                    while q:
                        r, c = q.popleft(); cells.append((r, c))
                        for nr, nc in ((r+1, c), (r-1, c), (r, c+1), (r, c-1)):
                            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and not seen[nr][nc]:
                                seen[nr][nc] = True; q.append((nr, nc))
                    M, best = to_matrix(cells), None
                    for base in (M, mirror(M)):
                        cur = base
                        for _ in range(4):
                            s = '/'.join(cur)
                            if best is None or s < best:
                                best = s
                            cur = rotate(cur)
                    shapes.add(best)
        return len(shapes)
```

**Độ phức tạp.** Mỗi đảo tốn `O(8 · B)` với `B` là diện tích khung bao (`B ≥ S`, có thể lớn hơn nhiều với đảo dạng đường chéo/zigzag, nhưng tổng vẫn bị chặn bởi `O(I · N)` thô; thực tế rất nhanh với lưới 50×50). Bộ nhớ `O(N)`.

## 8. TRACE từng bước

### 8.1. Bảng 8 phiên bản của đảo chữ L trong ví dụ 2

Đảo `{(0,0),(0,1),(0,2),(1,0)}` (hàng 3 ô, thêm một ô dưới đầu trái):

| Phép | Toạ độ sau biến đổi | Sau chuẩn hoá (sắp xếp) |
|---|---|---|
| `(x, y)` | (0,0),(0,1),(0,2),(1,0) | ((0,0),(0,1),(0,2),(1,0)) |
| `(x, -y)` | (0,0),(0,-1),(0,-2),(1,0) | ((0,0),(0,1),(0,2),(1,2)) |
| … | … | … |

Hàm `trace_canonical` in đủ cả 8 dòng và đánh dấu phiên bản nhỏ nhất.

### 8.2. `trace_canonical`

In [11]:
NAMES = ["(x, y)", "(x,-y)", "(-x,y)", "(-x,-y)", "(y, x)", "(y,-x)", "(-y,x)", "(-y,-x)"]

def trace_canonical(cells):
    print("đảo:", sorted(cells))
    versions = []
    for name, t in zip(NAMES, TRANSFORMS):
        pts = [t(r, c) for r, c in cells]
        r0 = min(p[0] for p in pts); c0 = min(p[1] for p in pts)
        v = tuple(sorted((p[0] - r0, p[1] - c0) for p in pts))
        versions.append((name, v))
    best = min(v for _, v in versions)
    for name, v in versions:
        pic = ' | '.join(draw_cells(list(v)))
        print(f"  {name:<8} -> {v}   hình: {pic}{'   <== nhỏ nhất' if v == best else ''}")
    return best

isl = find_islands(ex2)
c1 = trace_canonical(isl[0])
print()
c3 = trace_canonical([cells for cells in isl if len(cells) == 4][1])
print("\nhai chữ L có cùng dạng chuẩn?", c1 == c3)

đảo: [(0, 0), (0, 1), (0, 2), (1, 0)]
  (x, y)   -> ((0, 0), (0, 1), (0, 2), (1, 0))   hình: ### | #..   <== nhỏ nhất
  (x,-y)   -> ((0, 0), (0, 1), (0, 2), (1, 2))   hình: ### | ..#
  (-x,y)   -> ((0, 0), (1, 0), (1, 1), (1, 2))   hình: #.. | ###
  (-x,-y)  -> ((0, 2), (1, 0), (1, 1), (1, 2))   hình: ..# | ###
  (y, x)   -> ((0, 0), (0, 1), (1, 0), (2, 0))   hình: ## | #. | #.
  (y,-x)   -> ((0, 0), (0, 1), (1, 1), (2, 1))   hình: ## | .# | .#
  (-y,x)   -> ((0, 0), (1, 0), (2, 0), (2, 1))   hình: #. | #. | ##
  (-y,-x)  -> ((0, 1), (1, 1), (2, 0), (2, 1))   hình: .# | .# | ##

đảo: [(2, 1), (3, 1), (3, 2), (3, 3)]
  (x, y)   -> ((0, 0), (1, 0), (1, 1), (1, 2))   hình: #.. | ###
  (x,-y)   -> ((0, 2), (1, 0), (1, 1), (1, 2))   hình: ..# | ###
  (-x,y)   -> ((0, 0), (0, 1), (0, 2), (1, 0))   hình: ### | #..   <== nhỏ nhất
  (-x,-y)  -> ((0, 0), (0, 1), (0, 2), (1, 2))   hình: ### | ..#
  (y, x)   -> ((0, 0), (0, 1), (1, 1), (2, 1))   hình: ## | .# | .#
  (y,-x)   -> ((0, 0), (0, 1), (1

### 8.3. `trace_count` — đi qua từng đảo và tập dạng chuẩn

In [12]:
def trace_count(grid):
    shapes = {}
    for k, cells in enumerate(find_islands(grid)):
        cf = canonical(cells)
        status = "MỚI" if cf not in shapes else f"trùng với đảo #{shapes[cf]}"
        shapes.setdefault(cf, k)
        print(f"đảo #{k}: {len(cells)} ô, bắt đầu tại {min(cells)}; dạng chuẩn {cf} -> {status}")
    print("số loại đảo:", len(shapes))
    return len(shapes)

trace_count(ex2)
print()
ex3 = parse(["11011", "10001", "00000", "11011", "10001"])
show(ex3)
trace_count(ex3)

đảo #0: 4 ô, bắt đầu tại (0, 0); dạng chuẩn ((0, 0), (0, 1), (0, 2), (1, 0)) -> MỚI
đảo #1: 2 ô, bắt đầu tại (1, 4); dạng chuẩn ((0, 0), (0, 1)) -> MỚI
đảo #2: 4 ô, bắt đầu tại (2, 1); dạng chuẩn ((0, 0), (0, 1), (0, 2), (1, 0)) -> trùng với đảo #0
số loại đảo: 2

##.##
#...#
.....
##.##
#...#
đảo #0: 3 ô, bắt đầu tại (0, 0); dạng chuẩn ((0, 0), (0, 1), (1, 0)) -> MỚI
đảo #1: 3 ô, bắt đầu tại (0, 3); dạng chuẩn ((0, 0), (0, 1), (1, 0)) -> trùng với đảo #0
đảo #2: 3 ô, bắt đầu tại (3, 0); dạng chuẩn ((0, 0), (0, 1), (1, 0)) -> trùng với đảo #0
đảo #3: 3 ô, bắt đầu tại (3, 3); dạng chuẩn ((0, 0), (0, 1), (1, 0)) -> trùng với đảo #0
số loại đảo: 1


1

Ở `ex3`: bốn chữ L nhỏ ở bốn góc là ảnh của nhau qua các phép lật → chỉ **1** loại.

## 9. Code nộp LeetCode

Chọn **Cách 2** (dạng chuẩn bằng tuple toạ độ, DFS ngăn xếp).

In [13]:
from typing import List

class Solution:
    def numDistinctIslands2(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        T = [(1, 1, False), (1, -1, False), (-1, 1, False), (-1, -1, False),
             (1, 1, True), (1, -1, True), (-1, 1, True), (-1, -1, True)]
        shapes = set()
        for sr in range(m):
            for sc in range(n):
                if grid[sr][sc] and not seen[sr][sc]:
                    seen[sr][sc] = True
                    stack, cells = [(sr, sc)], []
                    while stack:
                        r, c = stack.pop()
                        cells.append((r, c))
                        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and not seen[nr][nc]:
                                seen[nr][nc] = True
                                stack.append((nr, nc))
                    best = None
                    for a, b, swap in T:
                        pts = [(b * c, a * r) if swap else (a * r, b * c) for r, c in cells]
                        r0 = min(p[0] for p in pts)
                        c0 = min(p[1] for p in pts)
                        v = tuple(sorted((x - r0, y - c0) for x, y in pts))
                        if best is None or v < best:
                            best = v
                    shapes.add(best)
        return len(shapes)

assert Solution().numDistinctIslands2(parse(["11000", "10000", "00001", "00011"])) == 1
assert Solution().numDistinctIslands2(parse(["11100", "10001", "01001", "01110"])) == 2
print("Ví dụ: OK")

Ví dụ: OK


## 10. Vì sao thuật toán đúng

**Nhóm 8 phép.** Tập `G` gồm 8 phép `(x, y) → (±x, ±y)` và `(±y, ±x)` là **nhóm đối xứng của hình vuông** (dihedral D4): hợp hai phép thuộc `G`, mỗi phép có phép ngược thuộc `G`, có phép đồng nhất. Mọi phép xoay 90/180/270 và mọi phép lật (ngang, dọc, cũng như qua đường chéo — là hợp của xoay và lật) đều thuộc `G`.

**Dạng chuẩn tách đúng các lớp.** Gọi `N(X)` là chuẩn hoá tịnh tiến + sắp xếp. `N(X) = N(Y)` ⇔ `X`, `Y` chỉ khác nhau một phép tịnh tiến. Với đảo `A`, đặt `V(A) = {N(g(A)) : g ∈ G}`.

- Nếu `B ≅ h(A)` (sai khác tịnh tiến) với `h ∈ G`: `V(B) = {N(g(h(A)))} = {N(g'(A)) : g' = g∘h}` = `V(A)` (vì `g ↦ g∘h` là song ánh trên `G`, và phép biến đổi tuyến tính giữ nguyên quan hệ "khác nhau một tịnh tiến"). ⇒ `min V(A) = min V(B)`.
- Nếu `min V(A) = min V(B) = N(g1(A)) = N(g2(B))` thì `g1(A)` và `g2(B)` khác nhau một tịnh tiến ⇒ `B ≅ g2⁻¹(g1(A))` ⇒ cùng loại.

Vậy số dạng chuẩn khác nhau = số loại đảo. **Tìm đảo** bằng DFS/BFS 4 hướng với `seen` đánh dấu khi đẩy vào → mỗi ô thuộc đúng một đảo.

**Cách 3** tương tự: `rotate` (xoay 90°) sinh 4 phép xoay; kết hợp `mirror` sinh 4 phép còn lại — đủ 8 phần tử của `G`; chuỗi ma trận khung bao (có dấu ngăn hàng) xác định duy nhất hình đã chuẩn hoá tịnh tiến.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. So từng cặp | `O(I² · 8 · B)` | `O(N)` |
| 2. Dạng chuẩn tuple | `O(8 · N log N)` | `O(N)` |
| 3. Dạng chuẩn chuỗi | `O(Σ_đảo 8 · B)` | `O(N)` |

(`N = m·n ≤ 2500`, `I` = số đảo, `B` = diện tích khung bao.)

Đo trên lưới 50×50 ngẫu nhiên (mật độ 0.4 → rất nhiều đảo nhỏ) và lưới "bàn cờ" (1250 đảo một ô — rất nhiều đảo nhưng chỉ một loại, nên so từng cặp chỉ phải so với một đại diện; trường hợp xấu cho Cách 1 là **nhiều đảo và nhiều loại** cùng lúc):

In [14]:
import random, time

random.seed(711)
rand_grid = [[1 if random.random() < 0.4 else 0 for _ in range(50)] for _ in range(50)]
checker = [[(r + c) % 2 for c in range(50)] for r in range(50)]
for gname, g in [("ngẫu nhiên 0.4", rand_grid), ("bàn cờ", checker)]:
    for name, fn in [("brute", distinct_islands_brute), ("canon", distinct_islands_canon), ("string", distinct_islands_string)]:
        t0 = time.perf_counter()
        r = fn(g)
        print(f"{gname:<15} {name:<7}: {r:>4} loại  {time.perf_counter() - t0:.3f}s")

ngẫu nhiên 0.4  brute  :   62 loại  0.003s
ngẫu nhiên 0.4  canon  :   62 loại  0.004s
ngẫu nhiên 0.4  string :   62 loại  0.003s
bàn cờ          brute  :    1 loại  0.007s
bàn cờ          canon  :    1 loại  0.007s
bàn cờ          string :    1 loại  0.008s


## 12. Bộ test

1. Ví dụ (theo hiểu biết về đề).
2. Edge case: lưới toàn 0; lưới toàn 1 (một đảo); các đảo 1 ô; chữ S và chữ Z (chỉ giống nhau khi cho phép **lật**); hình vuông 2×2 đối xứng hoàn toàn; đảo dạng đường chéo bậc thang.
3. Ngẫu nhiên có seed: mọi cách so với brute force.
4. **Tính chất bất biến:** xoay/lật/hoán vị cả lưới → đáp án không đổi.

In [15]:
METHODS = [distinct_islands_canon, distinct_islands_string, lambda g: Solution().numDistinctIslands2(g)]

def check(grid, expected=None):
    ref = distinct_islands_brute(grid)
    for f in METHODS:
        got = f([row[:] for row in grid])
        assert got == ref, (grid, got, ref)
    if expected is not None:
        assert ref == expected, (grid, ref, expected)
    return ref

check(ex1, 1)
check(ex2, 2)
check(parse(["0000", "0000"]), 0)
check(parse(["111", "111"]), 1)
check(parse(["10101", "00000", "10101"]), 1)                 # toàn đảo 1 ô
check(parse(["0110", "1100", "0000", "1100", "0110"]), 1)    # S và Z: giống nhau nhờ phép lật
check(parse(["11000", "11000", "00011", "00011"]), 1)        # hai hình vuông 2x2
check(parse(["1100", "0110", "0011"]), 1)                    # bậc thang (một đảo)
check(parse(["111", "000", "101", "101"]), 2)                # thanh ngang 3 ô vs hai thanh dọc 2 ô
check(ex3, 1)
print("Ví dụ + edge case: OK")

def rot_grid(g):
    return [list(r) for r in zip(*g[::-1])]

def flip_grid(g):
    return [row[::-1] for row in g]

random.seed(2030)
for _ in range(600):
    m, n = random.randint(1, 6), random.randint(1, 6)
    p = random.choice([0.3, 0.5, 0.7])
    g = [[1 if random.random() < p else 0 for _ in range(n)] for _ in range(m)]
    ans = check(g)
    assert distinct_islands_canon(rot_grid(g)) == ans
    assert distinct_islands_canon(flip_grid(g)) == ans
    assert distinct_islands_canon(rot_grid(rot_grid(flip_grid(g)))) == ans
print("600 lưới ngẫu nhiên: các cách khớp brute force; đáp án bất biến khi xoay/lật cả lưới")

Ví dụ + edge case: OK


600 lưới ngẫu nhiên: các cách khớp brute force; đáp án bất biến khi xoay/lật cả lưới


## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ xét 4 phép xoay, quên phép lật

In [16]:
def distinct_rotations_only(grid):
    shapes = set()
    for cells in find_islands(grid):
        shapes.add(min(normalize([t(r, c) for r, c in cells]) for t in
                       [TRANSFORMS[0], TRANSFORMS[3], TRANSFORMS[5], TRANSFORMS[6]]))   # chỉ 4 phép xoay
    return len(shapes)

sz = parse(["0110", "1100", "0000", "1100", "0110"])
show(sz)
print("sai:", distinct_rotations_only(sz), "| đúng:", distinct_islands_brute(sz))

.##.
##..
....
##..
.##.
sai: 2 | đúng: 1


**Vì sao sai:** chữ S và chữ Z không xoay được thành nhau, chỉ **lật** được. Đề cho phép cả lật → phải dùng đủ 8 phép.

### Lỗi 2: Quên chuẩn hoá tịnh tiến

In [17]:
def canonical_no_translate(cells):
    return min(tuple(sorted(t(r, c) for r, c in cells)) for t in TRANSFORMS)    # SAI: không trừ góc

def distinct_no_translate(grid):
    return len({canonical_no_translate(c) for c in find_islands(grid)})

print("sai:", distinct_no_translate(ex1), "| đúng:", distinct_islands_brute(ex1))

sai: 2 | đúng: 1


**Vì sao sai:** hai đảo cùng hình ở hai vị trí khác nhau có toạ độ khác nhau → tuple khác nhau. Phải trừ `(min r, min c)` **sau** mỗi phép biến đổi (vì phép biến đổi làm đổi vị trí khung bao).

### Lỗi 3: Chuẩn hoá tịnh tiến **trước** rồi mới biến đổi (không chuẩn hoá lại)

In [18]:
def canonical_translate_first(cells):
    base = normalize(cells)
    return min(tuple(sorted(t(r, c) for r, c in base)) for t in TRANSFORMS)    # SAI: toạ độ âm, chưa chuẩn hoá lại

def distinct_translate_first(grid):
    return len({canonical_translate_first(c) for c in find_islands(grid)})

bad = None
random.seed(3)
for _ in range(500):
    g = [[1 if random.random() < 0.5 else 0 for _ in range(5)] for _ in range(5)]
    if distinct_translate_first(g) != distinct_islands_brute(g):
        bad = g
        break
print("tìm được lưới phản ví dụ:", bad is not None)
if bad:
    show(bad)
    print("sai:", distinct_translate_first(bad), "| đúng:", distinct_islands_brute(bad))

tìm được lưới phản ví dụ: True
#..##
...#.
###.#
#..#.
#..##
sai: 4 | đúng: 3


**Vì sao sai:** sau phép `(x, y) → (-x, y)` toạ độ có thể âm và khung bao bị dời đi một lượng **phụ thuộc kích thước đảo**; hai đảo cùng loại có thể cho phiên bản lệch nhau. Luôn chuẩn hoá **sau** khi biến đổi.

### Lỗi 4: So sánh list ô **chưa sắp xếp**

Thứ tự DFS/BFS phụ thuộc điểm bắt đầu và hướng duyệt; cùng một hình có thể cho list theo thứ tự khác. Phải `sorted(...)` trước khi biến thành tuple (hoặc dùng `frozenset`).

## 14. Mẹo & nhận dạng pattern

- **"Đếm số hình dạng khác nhau dưới một nhóm biến đổi"** → dạng chuẩn = `min` trên mọi phần tử của nhóm; đếm bằng `set`.
- **Nhóm D4 (8 phép)** = `(±x, ±y)` và `(±y, ±x)` — nhớ công thức này là đủ.
- **Chuẩn hoá tịnh tiến**: trừ góc trên-trái (hoặc trừ điểm nhỏ nhất sau khi sắp xếp), làm **sau** mỗi phép biến đổi.
- **Đảo lớn** → DFS bằng ngăn xếp tường minh, tránh tràn đệ quy.
- Bài anh em: *Number of Distinct Islands* (694, chỉ tịnh tiến), *Number of Islands* (200), *Max Area of Island* (695), *Making A Large Island* (827).

## 15. Biến thể

1. **Chỉ tịnh tiến** (LeetCode 694) → dạng chuẩn = `normalize(cells)`, hoặc mã hoá đường đi DFS.
2. **Chỉ xoay, không lật** → dùng 4 phép xoay (đó chính là "Lỗi 1" — đúng cho biến thể này!).
3. **Đếm số đảo trong mỗi loại** → `Counter(canonical(cells) for cells in islands)`.
4. **Liên thông 8 hướng** → thêm 4 hướng chéo khi tìm đảo; phần dạng chuẩn giữ nguyên.
5. **Đảo 3D** (khối lập phương) → nhóm đối xứng 48 phần tử (hoặc 24 nếu chỉ xoay) — cùng ý tưởng.

Ví dụ biến thể 1 và 3:

In [19]:
from collections import Counter

def distinct_translation_only(grid):
    return len({normalize(cells) for cells in find_islands(grid)})

def count_per_shape(grid):
    return Counter(canonical(cells) for cells in find_islands(grid))

print("chỉ tịnh tiến, ex2:", distinct_translation_only(ex2), "| có xoay/lật:", distinct_islands_canon(ex2))
for shape, cnt in count_per_shape(ex3).items():
    print(cnt, "đảo có hình", draw_cells(list(shape)))

chỉ tịnh tiến, ex2: 3 | có xoay/lật: 2
4 đảo có hình ['##', '#.']


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng cặp | Ma trận khung bao; so với mọi đại diện qua 8 hướng | `O(I² · 8 · B)` | `O(N)` | Trọng tài khi test; ít đảo |
| 2. Dạng chuẩn tuple | `min` 8 tuple toạ độ đã chuẩn hoá; `set` | `O(N log N)` | `O(N)` | Lời giải chuẩn (khuyên dùng) |
| 3. Dạng chuẩn chuỗi | `min` 8 chuỗi ma trận (có dấu ngăn hàng); BFS | `O(Σ 8·B)` | `O(N)` | Khi thích làm việc với ma trận/`zip`; đảo "đặc" |

## 17. Bài tập

**Bài 1 (đoán output).** Số loại đảo của: `["101", "000", "101"]`; `["110", "011"]` (một đảo); `["1100", "0000", "0011", "0001"]`?

**Bài 2 (tính tay).** Viết 8 phiên bản đã chuẩn hoá của đảo `{(0,0),(1,0),(1,1)}` và chỉ ra dạng chuẩn.

**Bài 3 (điền TODO).** Hoàn thiện `my_normalize(points)` (trừ góc trên-trái và sắp xếp).

**Bài 4 (tự viết lại).** Viết lại `my_distinct(grid)` theo Cách 2.

**Bài 5 (mini challenge).** Viết `same_shape(cells_a, cells_b)` trả về `True` nếu hai đảo cùng loại **mà không dùng dạng chuẩn** (thử 8 phép trên `a`, so với `normalize(b)`), rồi đối chiếu với `canonical(a) == canonical(b)` trên nhiều cặp ngẫu nhiên.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_normalize(points):
    # TODO: r0 = min hàng, c0 = min cột; trả tuple(sorted((r - r0, c - c0) ...))
    pass

# Bài 4 — TODO
def my_distinct(grid):
    # TODO: tìm đảo (DFS ngăn xếp); canonical = min trên 8 phép của my_normalize; đếm bằng set
    pass

# Bài 5 — TODO
def same_shape(cells_a, cells_b):
    # TODO: any(normalize([t(r, c) for r, c in cells_a]) == normalize(cells_b) for t in TRANSFORMS)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tìm các đảo trước.
- Mức 2: bộ 3: một domino ngang `{(0,0),(0,1)}` và một chữ L 3 ô `{(2,2),(2,3),(3,3)}`.
- Mức 3: `1` (bốn đảo 1 ô), `1`, `2`.

**Bài 2.**
- Mức 1: áp từng công thức ở mục 3.3.
- Mức 2: chỉ có 4 hình khác nhau (chữ L 3 ô ở 4 góc khuyết).
- Mức 3: các hình: `((0,0),(1,0),(1,1))`, `((0,1),(1,0),(1,1))`, `((0,0),(0,1),(1,0))`, `((0,0),(0,1),(1,1))`; nhỏ nhất `((0,0),(0,1),(1,0))`.

**Bài 3.**
- Mức 1: `min(r for r, _ in points)`.
- Mức 2: trừ rồi `sorted`.
- Mức 3: trả `tuple(...)` để băm được.

**Bài 4.**
- Mức 1: dùng lại `find_islands`.
- Mức 2: 8 phép trong `TRANSFORMS`.
- Mức 3: `len({canon(c) for c in islands})`.

**Bài 5.**
- Mức 1: cần cùng số ô.
- Mức 2: thử 8 phép trên `a`.
- Mức 3: đối chiếu với dạng chuẩn trên các đảo sinh ngẫu nhiên.

## 19. Đáp án tham khảo

In [21]:
S = Solution()
# Bài 1
assert S.numDistinctIslands2(parse(["101", "000", "101"])) == 1
assert S.numDistinctIslands2(parse(["110", "011"])) == 1
assert S.numDistinctIslands2(parse(["1100", "0000", "0011", "0001"])) == 2

# Bài 2
cells = [(0, 0), (1, 0), (1, 1)]
vers = {normalize([t(r, c) for r, c in cells]) for t in TRANSFORMS}
assert vers == {((0, 0), (1, 0), (1, 1)), ((0, 1), (1, 0), (1, 1)),
                ((0, 0), (0, 1), (1, 0)), ((0, 0), (0, 1), (1, 1))}
assert canonical(cells) == ((0, 0), (0, 1), (1, 0))

# Bài 3
def my_normalize_ans(points):
    r0 = min(r for r, _ in points)
    c0 = min(c for _, c in points)
    return tuple(sorted((r - r0, c - c0) for r, c in points))

assert my_normalize_ans([(5, 7), (4, 7), (4, 8)]) == ((0, 0), (0, 1), (1, 0))
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4
def my_distinct_ans(grid):
    return len({min(my_normalize_ans([t(r, c) for r, c in cells]) for t in TRANSFORMS)
                for cells in find_islands(grid)})

# Bài 5
def same_shape_ans(a, b):
    if len(a) != len(b):
        return False
    nb = normalize(b)
    return any(normalize([t(r, c) for r, c in a]) == nb for t in TRANSFORMS)

random.seed(55)
pool = []
for _ in range(300):
    g = [[1 if random.random() < 0.55 else 0 for _ in range(5)] for _ in range(5)]
    assert my_distinct_ans(g) == distinct_islands_brute(g)
    pool.extend(find_islands(g))
pool = [c for c in pool if len(c) <= 6]
for _ in range(3000):
    a, b = random.choice(pool), random.choice(pool)
    assert same_shape_ans(a, b) == (canonical(a) == canonical(b))
print("Bài 4–5 OK (đối chiếu brute force và dạng chuẩn)")

Bài 4–5 OK (đối chiếu brute force và dạng chuẩn)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template dạng chuẩn dưới nhóm D4:**

```python
T = [(1,1,F),(1,-1,F),(-1,1,F),(-1,-1,F),(1,1,T),(1,-1,T),(-1,1,T),(-1,-1,T)]
def canon(cells):
    best = None
    for a, b, swap in T:
        pts = [(b*c, a*r) if swap else (a*r, b*c) for r, c in cells]
        r0 = min(x for x, _ in pts); c0 = min(y for _, y in pts)
        v = tuple(sorted((x - r0, y - c0) for x, y in pts))
        best = v if best is None or v < best else best
    return best
answer = len({canon(island) for island in islands})
```

**Checklist trước khi submit:**

- [ ] Đủ **8** phép (4 xoay + 4 lật), không chỉ 4 phép xoay.
- [ ] Chuẩn hoá tịnh tiến **sau** mỗi phép biến đổi.
- [ ] Sắp xếp ô trước khi tạo tuple (hoặc dùng `frozenset`).
- [ ] Liên thông **4 hướng**; DFS bằng ngăn xếp để tránh tràn đệ quy.
- [ ] Đánh dấu `seen` khi đẩy vào ngăn xếp/hàng đợi.
- [ ] Test: S/Z → 1 loại; ví dụ 1 → 1; ví dụ 2 → 2; lưới toàn 0 → 0.
- [ ] Tên method đúng: `numDistinctIslands2(self, grid)` (bài Premium — đối chiếu lại đề nếu có tài khoản).